# E10_Imitation — Behavioral Cloning и GAIL

**Урок:** 3.4 «Imitation Learning: Behavioral Cloning и GAIL» курса
[cyber-unity-learn](https://rl-cuber-unity-code.com/courses/3-4)
**Что реализуется:** запись демонстраций скриптовым экспертом, обучение
с учителем (BC) и состязательная имитация (GAIL) на одной задаче.
**Ожидаемый результат:** BC — **≥ 90 %** пройденных коридоров,
GAIL — **≥ 80 %** без использования функции награды.
**Время:** BC — около 20 секунд; GAIL — около 25 минут
(`QUICK_RUN = True` — около трёх минут).

## Задача, устроенная так, чтобы RL на ней буксовал

Змеевидный коридор 9 × 9 без развилок. От старта до выхода — **48 ходов**,
лимит времени — 120. Награда даётся только на выходе.

```
  ряд 8  ·  ·  ·  ·  ·  ·  ·  ·  G
  ряд 7  ·  █  █  █  █  █  █  █  █
  ряд 6  ·  ·  ·  ·  ·  ·  ·  ·  ·
  ряд 5  █  █  █  █  █  █  █  █  ·
  ряд 4  ·  ·  ·  ·  ·  ·  ·  ·  ·
  ряд 3  ·  █  █  █  █  █  █  █  █
  ряд 2  ·  ·  ·  ·  ·  ·  ·  ·  ·
  ряд 1  █  █  █  █  █  █  █  █  ·
  ряд 0  S  ·  ·  ·  ·  ·  ·  ·  ·
```

Измерено на билде: **случайная политика доходит до выхода в 0 из 104
эпизодов**. То есть на старте градиент RL тождественно нулевой, и ни
формирование награды (безопасной формы для лабиринта нет), ни учебный план
(шкалы сложности нет) не помогают.

Зато показать правильное поведение легко. В этом и весь смысл урока:
**формализовать награду часто труднее, чем продемонстрировать поведение.**

## Три метода на одной задаче

| Метод | Что берёт | Чем платит |
|---|---|---|
| **PPO** (`configs/E10_Imitation__ppo_discrete.yaml`) | награду среды | 600 000 шагов и низкий порог приёмки |
| **BC** | только записи эксперта, среду не трогает | требует эксперта; уязвим к сдвигу распределения |
| **GAIL** | записи эксперта + взаимодействие со средой, **без функции награды** | состязательное обучение, которое надо удерживать от расхождения |

## Требования

```powershell
python scripts\build_env.py E10_Imitation
```

Демонстрации уже лежат в репозитории (`data/demos/`, 8 КБ), поэтому раздел
BC работает **даже без билда**.

## 1. Окружение

In [ ]:
# --- 1. Окружение -------------------------------------------------------
import platform
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "python" / "labrl").is_dir():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError("не найден корень репозитория")
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "python"))

import numpy as np
import torch
import torch.nn as nn
import mlagents_envs

from labrl.utils.seeding import resolve_device, set_global_seed

SEED = 0
set_global_seed(SEED)
DEVICE = resolve_device("auto")

print(f"python        {platform.python_version()}")
print(f"torch         {torch.__version__}  CUDA {torch.version.cuda}")
print(f"mlagents_envs {mlagents_envs.__version__}")
print(f"устройство    {DEVICE}")
print(f"сид           {SEED}")

## 2. Конфигурация

Значения совпадают с `configs/E10_Imitation__bc.yaml`
и `configs/E10_Imitation__gail.yaml`.

Про `discriminator_learning_rate`: он **на порядок ниже**, чем у политики,
и это не осторожность, а необходимость. Слишком быстрый дискриминатор
выигрывает состязание, его точность уходит к 1.0, и градиент политике
обнуляется.

In [ ]:
QUICK_RUN = True   # True — быстрая проверка (~3 мин); False — полное обучение (~25 мин)

from labrl.algos.bc import BCConfig
from labrl.algos.gail import GAILConfig
from labrl.algos.ppo import PPOConfig
from labrl.eval.success import TERMINATED
from labrl.train.imitation import BCTrainConfig, GAILTrainConfig
from labrl.utils.schedules import LinearSchedule

DEMO_EPISODES = 100

# --- BC: «шаг» здесь мини-батч, а не шаг среды ---
BC_CFG = BCConfig(learning_rate=1e-3, batch_size=256, entropy_coef=0.0, max_grad_norm=1.0)
BC_TRAIN = BCTrainConfig(
    steps=2_000 if QUICK_RUN else 4_000,
    eval_every_steps=500,
    eval_episodes=30,
    holdout=0.2,             # без отложенной части «выучил» неотличимо от «запомнил»
    success_rule=TERMINATED,
)

# --- GAIL: полноценный RL-цикл, просто с другой наградой ---
GAIL_STEPS = 15_000 if QUICK_RUN else 150_000
POLICY_CFG = PPOConfig(
    gamma=0.99, gae_lambda=0.95, learning_rate=3e-4, clip_range=0.2,
    epochs=4, minibatch_size=512, value_coef=0.5, entropy_coef=0.01,
    max_grad_norm=0.5, normalize_advantage=True,
)
GAIL_CFG = GAILConfig(
    learning_rate=1e-4,          # на порядок ниже, чем у политики
    batch_size=256,
    updates_per_batch=2,
    label_smoothing=0.1,         # идеально уверенный дискриминатор глушит сигнал
    gradient_penalty=10.0,
    env_reward_weight=0.2,       # лечит склонность GAIL тянуть эпизод
)
GAIL_TRAIN = GAILTrainConfig(
    total_steps=GAIL_STEPS,
    rollout_steps=256,
    eval_every_steps=max(1, GAIL_STEPS // 8),
    eval_episodes=30,
    success_rule=TERMINATED,
)

# Расписание НЕ сжимается вместе с сокращённым прогоном (T-10).
LEARNING_RATE = LinearSchedule(start=3e-4, end=5e-5, decay_steps=150_000)

print(f"BC:   {BC_TRAIN.steps} мини-батчей")
print(f"GAIL: {GAIL_STEPS} шагов среды, обновлений: {GAIL_STEPS // GAIL_TRAIN.rollout_steps}")

## 3. Подключение среды и демонстрации

Эксперт — скриптовое правило: **не возвращайся туда, откуда пришёл; иди
в любую свободную клетку**. В коридоре без развилок этого достаточно.

Ключевое требование к нему: **он действует только по наблюдению агента**
(индексы 7–10 — стены вокруг, индекс 12 — предыдущий ход). Если бы эксперт
пользовался знанием, недоступным агенту, имитация училась бы невозможному:
политика идеально повторяла бы записи и разваливалась в среде, потому что
нужного признака во входе просто нет. Это проверено тестом
`test_expert_uses_only_the_agent_observation`.

In [ ]:
from labrl.envs.demos import Demonstrations, corridor_expert, record_demonstrations
from labrl.envs.registry import get as get_env
from labrl.envs.unity_env import open_unity_env
from labrl.envs.vec_unity_env import VecUnityEnv

ENV_ID = "E10_Imitation"
DEMO_PATH = REPO_ROOT / "data" / "demos" / f"{ENV_ID}_expert.npz"
BUILD = get_env(ENV_ID).build_path

handle = vec = None
if BUILD.is_file():
    handle = open_unity_env(ENV_ID, build_path=BUILD, seed=SEED,
                            time_scale=20.0, no_graphics=True, num_areas=8)
    vec = VecUnityEnv(handle)
    vec.reset()
    print(handle.describe())
    print(f"\nарен: {vec.num_envs}")
elif not DEMO_PATH.is_file():
    raise FileNotFoundError(
        f"нет ни билда {BUILD}, ни демонстраций {DEMO_PATH}\n"
        f"Соберите среду: python scripts/build_env.py {ENV_ID}")
else:
    print(f"билда нет — BC пойдёт на готовых демонстрациях {DEMO_PATH.name};\n"
          "разделы с оценкой в среде и GAIL будут пропущены")

if DEMO_PATH.is_file():
    demos = Demonstrations.load(DEMO_PATH)
else:
    demos = record_demonstrations(vec, corridor_expert, episodes=DEMO_EPISODES)
    demos.save(DEMO_PATH)

print("\n" + demos.describe())
print("действий по направлениям (север/юг/восток/запад):",
      demos.action_histogram().tolist())
print(f"пар на эпизод: {len(demos) / max(demos.episodes, 1):.1f} "
      f"(кратчайший путь — 48 ходов)")

## 4. Инспекция пространств

Наблюдение здесь целиком векторное — лучевого сенсора нет намеренно:
он дал бы ту же информацию расплывчато и заставил бы эксперта
интерпретировать длины лучей.

In [ ]:
OBS_DIM = demos.obs_dim
BRANCHES = (4,)

print(f"obs_dim: {OBS_DIM}, веток: {BRANCHES}")
if vec is not None:
    # Значения из строки-контракта ENV_SPEC.md:
    # <!-- validator: obs_size=13; discrete_branches=4; continuous_size=0; max_step=120 -->
    assert [tuple(s) for s in vec.obs_shapes] == [(13,)], vec.obs_shapes
    assert vec.single_obs_dim == 13
    assert tuple(vec.action_spec.discrete_branches) == BRANCHES
    assert vec.action_spec.continuous_size == 0
    print("размерности совпадают с ENV_SPEC.md")

print("\nсостав наблюдения:")
print("  [0:2]  клетка агента          [2:4]  клетка выхода")
print("  [4:6]  смещение до выхода     [6]    манхэттенское расстояние")
print("  [7:11] стены: С / Ю / В / З   [11]   доля времени")
print("  [12]   предыдущий ход / 3")
print("\nэксперт читает ТОЛЬКО [7:11] и [12] — то же, что видит агент")

## 5. Нейросеть

Та же категориальная политика, что у RL-методов. Это не совпадение:
BC часто используют как **инициализацию** для последующего RL, и сеть
должна быть той же.

In [ ]:
from labrl.nets.categorical_policy import MultiBranchCategoricalPolicy


class MyPolicy(MultiBranchCategoricalPolicy):
    """Политика: obs (B, 13) -> логиты (B, 4)."""

    def __init__(self, obs_dim: int, branches: tuple[int, ...], hidden: int = 128):
        super().__init__(obs_dim, branches, (hidden, hidden), "relu")


bc_policy = MyPolicy(OBS_DIM, BRANCHES)

with torch.no_grad():
    assert bc_policy(torch.zeros(3, OBS_DIM)).shape == (3, sum(BRANCHES))

print(bc_policy)
print(f"\nпараметров: {sum(p.numel() for p in bc_policy.parameters()):,}")

## 6. Теория: две задачи, которые решают BC и GAIL

### BC — обычная классификация

$$L(\theta) = -\,\mathbb{E}_{(s,a)\sim D}\big[\log \pi_\theta(a \mid s)\big]$$

Ни награды, ни ценности, ни исследования. Правильный ответ известен
на каждом примере сразу — отсюда и скорость.

### GAIL — состязание

$$L_D = -\,\mathbb{E}_{\text{эксперт}}\big[\log D(s,a)\big]
       -\,\mathbb{E}_{\pi}\big[\log(1 - D(s,a))\big],
\qquad r_{\text{GAIL}}(s,a) = -\log\big(1 - D(s,a)\big)$$

Форма награды выбрана не произвольно: $-\log(1-D)$ не ограничена сверху
и даёт сильный градиент ровно тогда, когда политика уже похожа на эксперта,
тогда как $\log D$ в этот момент насыщается.

Обратная сторона — **награда всегда положительна**, и у агента появляется
мотив тянуть эпизод. В средах «дойди до цели» это лечится подмешиванием
награды среды (`env_reward_weight`).

### Про сдвиг распределения — и почему здесь его не видно

Учебная слабость BC: он учится на состояниях, которые посещал эксперт,
и ошибка уводит туда, где его никто не учил; потеря растёт как
$O(\varepsilon T^2)$ (Ross & Bagnell, 2010).

**На этой среде такого не происходит, и это измеренный факт.** Коридор
без развилок: эксперт проходит все 49 клеток маршрута, и сходить с него
некуда. Это не опровергает теорему, а уточняет область её применимости:
$O(\varepsilon T^2)$ — верхняя граница, достижимая только там, где ошибка
выводит в незнакомую область.

Тогда зачем здесь GAIL? Ради другого свойства: он **вообще не пользуется
функцией награды**. Сравнение GAIL и PPO на одной среде показывает, сколько
стоит отказ от размеченной награды.

Ниже — обновление BC, тот самый код, что выполняется при обучении.

In [ ]:
import inspect

from labrl.algos.bc import BC

print(inspect.getsource(BC.update))

## 7. Обучение: BC

In [ ]:
from labrl.logging.run_dir import create_run_dir
from labrl.logging.tb_logger import TBLogger
from labrl.train.imitation import train_bc

bc = BC(bc_policy, BC_CFG, device=DEVICE, seed=SEED)
bc_run = create_run_dir(ENV_ID, "bc", SEED)
bc_logger = TBLogger(bc_run.tb)

print(f"каталог прогона: {bc_run.root}\n")

bc_result = train_bc(
    demos=demos, algo=bc, logger=bc_logger, vec=vec, cfg=BC_TRAIN, seed=SEED,
    on_eval=lambda step, r, s: print(f"  шаг {step:>5}: награда {r:>7.3f}, пройдено {s:.0%}"),
)

bc_missing = bc_logger.missing_required_tags()
bc_logger.dump_metrics(bc_run.metrics_json,
                       extra={"seed": SEED, "missing_required_tags": list(bc_missing)})
bc_logger.close()

step, train_accuracy, holdout_accuracy = bc_result.accuracy_history[-1]
print(f"\nточность на обучающих парах:  {train_accuracy:.1%}")
print(f"точность на отложенных парах: {holdout_accuracy:.1%}")
if vec is not None:
    print(f"пройдено коридоров:           {bc_result.last_eval_success:.0%}")
print(f"время: {bc_result.wall_time:.1f} с")
print(f"обязательные теги схемы 11.2 без данных: {bc_missing or 'нет'}")

### Две метрики, которые легко перепутать

`Custom/Holdout Accuracy` — доля пар, на которых политика выбирает то же,
что эксперт. `Eval/Success Rate` — доля эпизодов, доведённых до цели.

Первая систематически **завышает** вторую: 2 % ошибок на 48 ходах почти
наверняка случаются хотя бы раз за эпизод. Приёмка идёт по второй.

Отложенная часть обязательна: точность на обучающих парах у достаточно
большой сети доходит до единицы в любом случае, и без неё «BC выучил
эксперта» неотличимо от «BC запомнил записи».

In [ ]:
import matplotlib
matplotlib.use("Agg")   # ноутбук должен исполняться и без графического бэкенда
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))

steps = [s for s, _, _ in bc_result.accuracy_history]
train_acc = [a for _, a, _ in bc_result.accuracy_history]
hold_acc = [h for _, _, h in bc_result.accuracy_history]

axes[0].plot(steps, train_acc, marker="o", label="обучающие пары")
axes[0].plot(steps, hold_acc, marker="s", label="отложенные пары")
axes[0].set_ylim(-0.05, 1.05)
axes[0].set_title("Точность имитации эксперта")
axes[0].set_xlabel("мини-батч"); axes[0].grid(alpha=0.3); axes[0].legend()

if bc_result.eval_history:
    eval_steps = [s for s, _, _ in bc_result.eval_history]
    eval_success = [v for _, _, v in bc_result.eval_history]
    axes[1].plot(eval_steps, eval_success, marker="o", color="tab:green")
    axes[1].axhline(0.9, color="k", ls="--", lw=1, label="критерий приёмки")
    axes[1].axhline(0.0, color="tab:red", ls=":", lw=1, label="случайная политика")
    axes[1].set_ylim(-0.05, 1.05)
    axes[1].legend()
else:
    axes[1].text(0.5, 0.5, "билда нет — оценка в среде пропущена",
                 ha="center", va="center")
axes[1].set_title("Eval/Success Rate — мера качества")
axes[1].set_xlabel("мини-батч"); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

BC_THRESHOLD = 0.90   # блок success_criteria конфига
if vec is None:
    print("билда нет: критерий приёмки не проверяется")
elif QUICK_RUN:
    print(f"быстрый прогон: критерий приёмки ({BC_THRESHOLD:.0%}) не проверяется")
else:
    assert bc_result.last_eval_success >= BC_THRESHOLD, (
        f"не достигнут критерий приёмки: {bc_result.last_eval_success:.0%}")
    print(f"критерий приёмки достигнут: {bc_result.last_eval_success:.0%}")

## 8. Обучение: GAIL

Здесь начинается настоящий RL-цикл — просто награду выдаёт дискриминатор,
а не среда. `env_reward_weight = 0.2` означает, что 80 % награды приходит
от дискриминатора.

**Главный график этого раздела — `Custom/Discriminator Accuracy`.**
0.5 означает «политика неотличима от эксперта» (цель метода); 1.0 —
«дискриминатор выиграл состязание, сигнал политике пропал».

In [ ]:
from labrl.algos.gail import GAIL, Discriminator
from labrl.algos.ppo_discrete import PPODiscrete
from labrl.nets.mlp import MLPValueNetwork
from labrl.train.imitation import train_gail

gail_result = None
if vec is None:
    print("билда нет — раздел GAIL пропущен")
else:
    gail_policy = MyPolicy(OBS_DIM, BRANCHES)
    gail_value = MLPValueNetwork(OBS_DIM, (128, 128), "relu")
    policy = PPODiscrete(gail_policy, gail_value, POLICY_CFG, device=DEVICE, seed=SEED)

    discriminator = Discriminator(OBS_DIM, BRANCHES, (128, 128), "tanh")
    gail = GAIL(discriminator, demos, GAIL_CFG, device=DEVICE, seed=SEED)

    gail_run = create_run_dir(ENV_ID, "gail", SEED)
    gail_logger = TBLogger(gail_run.tb)
    print(f"каталог прогона: {gail_run.root}\n")

    gail_result = train_gail(
        vec=vec, algo=policy, gail=gail, logger=gail_logger, cfg=GAIL_TRAIN, seed=SEED,
        lr_schedule=LEARNING_RATE,
        on_eval=lambda step, r, s: print(
            f"  шаг {step:>7}: награда среды {r:>7.3f}, пройдено {s:.0%}"),
    )

    gail_missing = gail_logger.missing_required_tags()
    gail_logger.dump_metrics(gail_run.metrics_json,
                             extra={"seed": SEED, "missing_required_tags": list(gail_missing)})
    gail_logger.close()

    print(f"\nпройдено коридоров: {gail_result.last_eval_success:.0%}")
    print(f"время: {gail_result.wall_time:.1f} с")
    print(f"обязательные теги схемы 11.2 без данных: {gail_missing or 'нет'}")

## 9. Сравнение

Сводка того, что стоит каждый метод. Числа PPO берутся из отдельного
прогона — он слишком долгий для ноутбука:

```powershell
python scripts\train.py --config configs\E10_Imitation__ppo_discrete.yaml --all-seeds
```

In [ ]:
rows = [
    ("случайная политика", "—", "0 %", "измерено: 0 из 104 эпизодов"),
    ("BC", f"{BC_TRAIN.steps} мини-батчей",
     f"{bc_result.last_eval_success:.0%}" if vec is not None else "—",
     "среду при обучении не трогает"),
]
if gail_result is not None:
    rows.append(("GAIL", f"{GAIL_STEPS} шагов среды",
                 f"{gail_result.last_eval_success:.0%}",
                 "функцию награды не использует"))
rows.append(("PPO", "600 000 шагов среды", "см. отдельный прогон",
             "только награда среды"))

width = max(len(r[0]) for r in rows)
print(f"{'метод':<{width}}  {'бюджет':>22}  {'пройдено':>10}  примечание")
print("-" * (width + 60))
for name, budget, result, note in rows:
    print(f"{name:<{width}}  {budget:>22}  {result:>10}  {note}")

## 10. Экспорт в ONNX и возврат в Unity

Экспортируется политика. У BC это вся модель целиком; у GAIL —
только актор: дискриминатору в Unity взять неоткуда пару
«наблюдение + действие».

In [ ]:
import onnxruntime as ort

from labrl.export.onnx_export import ActionSpecLite, export_policy_to_onnx
from labrl.export.onnx_verify import verify_onnx_model

if vec is None:
    print("билда нет — экспорт пропущен")
else:
    # Наблюдения из реального распределения среды (требование 10.5), а не шум.
    collected = []
    obs_now = vec.reset()
    while sum(len(c) for c in collected) < 64:
        collected.append(obs_now[0].copy())
        obs_now = vec.step(bc.deterministic_action(vec.flatten_obs(obs_now))).obs
    sample = [np.concatenate(collected, axis=0)[:64]]

    spec = ActionSpecLite(discrete_branches=BRANCHES, continuous_size=0)
    policy_module = bc.policy_module()
    onnx_path = export_policy_to_onnx(policy_module, spec, vec.obs_shapes,
                                      bc_run.onnx / "policy.onnx", strategy="categorical")

    verification = verify_onnx_model(onnx_path, policy_module, spec, vec.obs_shapes,
                                     sample_obs=sample)
    print(verification.report())
    verification.raise_if_failed()

    session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    graph_action = session.run(
        ["deterministic_discrete_actions"],
        {"obs_0": sample[0],
         "action_masks": np.ones((64, sum(BRANCHES)), dtype=np.float32),
         "recurrent_in": np.zeros((64, 1, 0), dtype=np.float32)},
    )[0]
    python_action = bc.deterministic_action(sample[0])
    mismatch = int((graph_action != python_action).sum())
    print(f"\nрасхождений действия ONNX и Python: {mismatch} из {python_action.size}")
    assert mismatch == 0, "оценка в Python и граф расходятся"

    if not QUICK_RUN:
        # Модель проекта обновляется только полным прогоном (T-14).
        model_dest = (REPO_ROOT / "unity" / "MLAgentsLab" / "Assets" / "Envs" / ENV_ID /
                      "Models" / f"{ENV_ID}_bc.onnx")
        model_dest.parent.mkdir(parents=True, exist_ok=True)
        model_dest.write_bytes(onnx_path.read_bytes())
        print(f"модель скопирована в проект Unity: {model_dest.relative_to(REPO_ROOT)}")
    else:
        print("быстрый прогон: модель в проект Unity НЕ копируется (T-14)")

    vec.handle.close()

### Вручную (чтобы увидеть глазами)

1. Открыть `unity/MLAgentsLab` в Unity 6000.5.4f1.
2. Открыть `Assets/Envs/E10_Imitation/Scenes/E10_Imitation.unity`.
3. Выделить модель в Project → **Tools → RL → Check ONNX Contract**.
4. У агента `Agent`: `Model` = импортированная модель,
   `Behavior Type` = `Inference Only`, галочка `Deterministic Inference`.
5. Play: агент проходит змейку от левого нижнего угла к правому верхнему.

### Ручная проверка среды с клавиатуры

`Behavior Type` = `Heuristic Only`, стрелки или **WASD** — одно нажатие,
один ход. Пройти коридор руками полезно: сразу видно, почему случайная
политика этого не делает.

### Если что-то идёт не так

| Симптом | Где искать |
|---|---|
| точность BC высокая, а коридоры не проходятся | сдвиг распределения либо эксперт видит больше агента |
| `Discriminator Accuracy` уходит к 1.0 | дискриминатор выиграл состязание: снизить его `learning_rate` |
| у GAIL эпизоды всё длиннее, до цели не доходит | награда `−log(1−D)` положительна: поднять `env_reward_weight` |

Общий список причин расхождения Python и Unity — в
`docs/07_TROUBLESHOOTING.md`.